In [1]:
pip install pandas statsmodels linearmodels

Note: you may need to restart the kernel to use updated packages.


In [3]:
df = pd.read_excel("df_final.xlsx")

In [4]:
import pandas as pd
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

# Carga los datos (asegúrate de usar el path correcto a tu archivo CSV si lo exportaste)
# df = pd.read_csv("ruta/a/tu/archivo.csv")

# Simulación de cómo debería lucir la carga de tu DataFrame:
# Ya tienes columnas como refAreaID (país) y timePeriod

# 1. Asegurar tipos de datos correctos
df['timePeriod'] = pd.to_datetime(df['timePeriod'])

# 2. Establecer el índice para panel: (país, trimestre)
df = df.set_index(['refAreaID', 'timePeriod'])

# 3. Eliminar filas con valores faltantes (opcional pero recomendable)
df = df.dropna()

# 4. Definir variables independientes y dependiente
y = df['Consumption_it']
X = df[['Fiscal_Stimulus_it', 'Inflation_it', 'Capital_Flight_it', 'Unemployment_it', 'Interest_Rate_it']]

# 5. Agregar constante
X = sm.add_constant(X)

# 6. Ajustar modelo con efectos fijos por país (entity_effects)
model = PanelOLS(y, X, entity_effects=True)
results = model.fit(cov_type='clustered', cluster_entity=True)

# 7. Mostrar resultados
print(results.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:         Consumption_it   R-squared:                        0.8080
Estimator:                   PanelOLS   R-squared (Between):              0.0228
No. Observations:                  52   R-squared (Within):               0.8080
Date:                Fri, May 02 2025   R-squared (Overall):              0.0521
Time:                        00:48:23   Log-likelihood                   -532.62
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      37.882
Entities:                           2   P-value                           0.0000
Avg Obs:                       26.000   Distribution:                    F(5,45)
Min Obs:                       24.000                                           
Max Obs:                       28.000   F-statistic (robust):         -1.529e+17
                            

In [6]:
# Extract results into a DataFrame
summary_df = pd.DataFrame({
    "Variable": results.params.index,
    "Coefficient": results.params.values,
    "Std. Error": results.std_errors,
    "t-Statistic": results.tstats,
    "p-Value": results.pvalues,
    "95% CI Lower": results.conf_int().iloc[:, 0],
    "95% CI Upper": results.conf_int().iloc[:, 1],
})

# Round for readability
summary_df = summary_df.round(2)

# Display as table
print(summary_df)

# Optional: export to CSV for use in Word or Excel
summary_df.to_csv("regression_results.csv", index=False)


                              Variable  Coefficient  Std. Error  t-Statistic  \
const                            const    104914.68    13229.85         7.93   
Fiscal_Stimulus_it  Fiscal_Stimulus_it   -556232.09    31793.14       -17.50   
Inflation_it              Inflation_it    186646.54    69487.15         2.69   
Capital_Flight_it    Capital_Flight_it     41936.46    63816.17         0.66   
Unemployment_it        Unemployment_it     -2760.38      483.46        -5.71   
Interest_Rate_it      Interest_Rate_it      1290.15      237.65         5.43   

                    p-Value  95% CI Lower  95% CI Upper  
const                  0.00      78268.39     131560.96  
Fiscal_Stimulus_it     0.00    -620266.75    -492197.42  
Inflation_it           0.01      46692.23     326600.86  
Capital_Flight_it      0.51     -86595.89     170468.82  
Unemployment_it        0.00      -3734.12      -1786.63  
Interest_Rate_it       0.00        811.50       1768.81  
